In [4]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import os
import torch.nn.functional as F

from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import confusion_matrix, roc_auc_score


In [5]:
X_train = np.load("../artifacts/data/X_train.npy")
X_test  = np.load("../artifacts/data/X_test.npy")

y_train = np.load("../artifacts/data/y_train.npy")
y_test  = np.load("../artifacts/data/y_test.npy")


In [6]:
#busca pela assinatura digital
#busca por similaridade

class Encoder(nn.Module):
    def __init__(self, input_dim, embedding_dim=64):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, embedding_dim)
        )

    def forward(self, x):
        z = self.net(x)
        return F.normalize(z, p=2, dim=1)


In [7]:
#sensor pré-treinado

device = "cuda" if torch.cuda.is_available() else "cpu"

encoder = Encoder(X_train.shape[1]).to(device)

encoder.load_state_dict(
    torch.load("../artifacts/models/encoder_contrastive.pth", map_location=device)
)

encoder.eval()

for param in encoder.parameters():
    param.requires_grad = False


In [8]:
#paro de trabalhar com dados brutos e começo com espaço latente
#transformo cada conexão em uma coordenada matemática única
#antes: 194 colunas com dados heterogêneos
#agora: 64 colunas de números normalizados que capturam a semântica de segurança
#isso é melhor que trabalhar com dados originais
#prepara o modelo para aprender continuamente
#detecta anomalias se um ponto cair na região Z
#velocidade de busca
#redução de ruído

def generate_embeddings(X, model, batch_size=512):
    model.eval()
    embeddings = []

    loader = DataLoader(
        TensorDataset(torch.tensor(X, dtype=torch.float32)),
        batch_size=batch_size
    )

    with torch.no_grad():
        for (xb,) in loader:
            xb = xb.to(device)
            z = model(xb)
            embeddings.append(z.cpu().numpy())

    return np.vstack(embeddings)


Z_train = generate_embeddings(X_train, encoder)
Z_test  = generate_embeddings(X_test, encoder)

print(Z_train.shape, Z_test.shape)


(175341, 64) (82332, 64)


In [9]:
#aqui há a tomada de decisão final
#pacote de rede é convertido em um embedding (vetor de 64 números)
#presença de velocidade e eficiência 
#EmbeddingClassifier processa o vetor quase que instantaneamente. Sistema 
#toma decisões em microsegundos, evitando não causar latência na rede
#aqui conecto a segurança moderna com a eficiência operacional

class EmbeddingClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze()


In [10]:
#fluxo de detecção final (pipeline):
#captura: pacote de rede chega (194 características)
#abstração (encoder): encoder transforma o pacote em uma coordenada gerográfica (64 dimensões)
#veredito (classifier): classificador olha pra coordenada e diz: "naquela região, 99% das vezes é um ataque"

Xtr = torch.tensor(Z_train, dtype=torch.float32)
ytr = torch.tensor(y_train, dtype=torch.float32)

train_loader = DataLoader(
    TensorDataset(Xtr, ytr),
    batch_size=256,
    shuffle=True
)

model = EmbeddingClassifier(Z_train.shape[1]).to(device)

positivos = np.sum(y_train == 1)
negativos = np.sum(y_train == 0)

pos_weight = torch.tensor([negativos / positivos]).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = optim.Adam(model.parameters(), lr=0.001)


In [11]:
epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()

        logits = model(xb)

        loss = criterion(logits.view(-1, 1), yb.view(-1, 1)) #alinhamento com BCEWithLogitsLoss

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f}")

Epoch 1/10 - Loss: 0.1113
Epoch 2/10 - Loss: 0.0804
Epoch 3/10 - Loss: 0.0782
Epoch 4/10 - Loss: 0.0771
Epoch 5/10 - Loss: 0.0765
Epoch 6/10 - Loss: 0.0761
Epoch 7/10 - Loss: 0.0757
Epoch 8/10 - Loss: 0.0754
Epoch 9/10 - Loss: 0.0753
Epoch 10/10 - Loss: 0.0750


In [21]:
#com algumas pequenas alterações nos valores, o modelo segue equilibrado e pronto
#para o aprendizado contínuo

model.eval()

with torch.no_grad():
    logits = model(torch.tensor(Z_test, dtype=torch.float32).to(device))
    probs = torch.sigmoid(logits).cpu().numpy()

threshold = 0.5

y_pred = (probs >= threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

recall = tp / (tp + fn)
fpr    = fp / (fp + tn)
auc    = roc_auc_score(y_test, probs)

print(f"Recall : {recall:.2%}") #sensibildiade: porcentagem total que o modelo conseguiu pegar
print(f"FPR    : {fpr:.2%}") #alarme falso
print(f"ROC AUC: {auc:.3}") #de 0 a 1, as habilidades do modelo em separar as duas classes



Recall : 94.45%
FPR    : 17.63%
ROC AUC: 0.974


In [13]:
np.save("../artifacts/data/Z_train.npy", Z_train)
np.save("../artifacts/data/Z_test.npy", Z_test)

print("Embeddings salvos.")


Embeddings salvos.


In [14]:
os.makedirs("../artifacts/models", exist_ok=True)

torch.save(
    model.state_dict(),
    "../artifacts/models/classifier_embeddings.pth"
)

print("Classificador salvo.")


Classificador salvo.
